# N005 · 列表、索引与对象引用

**目标：** 正确操作列表并区分原地修改和创建副本。

**先修：** N004。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 索引与切片；append/pop；浅拷贝；二维列表别名；可变对象。

**配套讲解来源：** [同名逐章意见](../../comment/005_lists_mutation_aliasing.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两道列表操作题，同时讲透一个 Python 新手最容易栽的坑：**列表里存的是引用，改一个可能牵动一片**。

**问题 1（数组拼接）**：给你一个长度 n 的数组 nums，返回一个长度 2n 的数组 ans，前 n 个元素和后 n 个元素都是 nums 的原样拷贝，即 ans = nums + nums。具体例子：输入 `nums=[1,2,1]`，输出 `[1,2,1,1,2,1]`——把原数组原封不动写两遍拼起来。附加要求是：不能把输入数组改坏。

**问题 2（构造全零矩阵）**：给定行数 rows 和列数 cols，造一个 rows×cols 的二维列表，里面全是 0。这个任务本身很 trivial，真正的难点是：**你必须造出 rows 个彼此独立的行**。经典错误写法是 `[[0]*cols]*rows`，它看起来没问题，实际上是把**同一行引用**重复了 rows 遍。具体例子：`bad = [[0]*3]*2` 之后执行 `bad[0][0] = 7`，你会发现 `bad` 变成了 `[[7,0,0],[7,0,0]]`——明明只改了第 0 行，第 1 行也跟着变了，因为两行根本是同一个对象。

## 2. 基础知识：先读懂这些词

- **对象引用（reference）**：Python 的变量和列表元素存的不是"东西本身"，而是"东西的地址"。赋值 `b = a` 只是把地址抄了一份，a 和 b 指向同一个列表。用 `a is b` 可以检查两个名字是不是指同一个对象。
- **可变对象（mutable）**：列表创建之后还能原地改内容（改元素、加元素），就叫可变对象。整数、字符串是不可变的——"改"它们其实是造了新对象。
- **原地修改（in-place mutation）vs 创建副本**：`lst.append(x)`、`lst[0]=7` 是原地修改，改动会波及所有指向这个列表的名字；`lst + lst`、`lst[:]` 则是创建新列表，原列表不动。写函数时要明确自己是哪一种，别让调用方莫名其妙被改了数据。
- **索引与切片**：`lst[0]` 取第 0 个元素（下标从 0 开始）；`lst[0][0]` 先取第 0 行再取该行第 0 个元素，这是访问二维列表的写法。切片 `lst[:]` 可以复制一份浅拷贝。
- **append / pop**：`append(x)` 在列表尾部原地追加一个元素；`pop()` 从尾部原地弹出一个元素。它们是"原地修改"的典型代表。
- **浅拷贝（shallow copy）**：只复制外层容器、不递归复制内部对象的拷贝。`lst[:]` 复制了列表壳子，但壳子里装的引用还是原来那些——如果里面嵌着列表，内层仍然是共享的。深拷贝（`copy.deepcopy`）才会连内层一起复制。
- **别名（aliasing）**：两个名字指向同一个对象，互相是对方的"别名"。`[[0]*3]*2` 就是制造别名的典型事故：两行互为别名，动一行全体遭殃。
- **列表推导式（list comprehension）**：`[[0 for _ in range(cols)] for _ in range(rows)]` 这种 `[表达式 for 变量 in 可迭代]` 写法。关键性质：外层推导式每循环一轮就执行一次内层表达式，所以内层列表会被**重新创建** rows 次，每次都是新对象。
- **`+` 拼接**：`nums + nums` 创建并返回一个全新列表，内容是两个操作数按顺序接起来。它不改任何一方。

## 3. 思路推导：从小例子开始

**思路 A：数组拼接。**

- **Step 1：想清楚"要新列表还是要原地改"。** 题目要求返回长度 2n 的新数组，而且不应该弄坏输入，所以我们选择"创建新列表"这条路。
- **Step 2：用 `+` 一行完成。** `nums + nums` 会让 Python 新建一个列表，先把左边 nums 的元素按顺序抄进去，再把右边 nums 的元素接在后面。原 nums 完全不动。
- **Step 3：验证身份。** 返回值 `b is not a` 应该成立（新对象），且 `a` 的内容保持 `[1,2,1]` 不变。

**手算演示一轮（nums=[1,2,1]）**：`+` 从左操作数抄起：抄 1、抄 2、抄 1，此时新列表是 `[1,2,1]`；接着从右操作数抄：再抄 1、2、1，新列表变成 `[1,2,1,1,2,1]`，长度 6 = 2n。整个过程中原 nums 的地址和内容都没被碰过。

**思路 B：全零矩阵。**

- **Step 1：认清错误写法为什么错。** `[[0]*cols]*rows` 分两步执行：内层 `[0]*cols` 先造出一行 `[0,0,...,0]`；外层 `*rows` 把**这一行的引用**复制 rows 遍。于是矩阵里所有行都是同一个对象，`bad[0] is bad[1]` 为 True。
- **Step 2：改用嵌套推导式。** 外层 `for _ in range(rows)` 每轮都会重新执行一次内层 `[0 for _ in range(cols)]`，也就是每轮都新建一行。这样 rows 个行是 rows 个不同的对象，`good[0] is good[1]` 为 False。
- **Step 3：挡住负数维度。** rows 或 cols 为负没有意义，入口直接抛 ValueError。
- **Step 4：验证独立性。** 改 `good[0][0]` 后检查 `good[1][0]` 是否还是 0。

**手算演示一轮（“运行示例”部分 的对照实验）**：“运行示例”部分 用 `show_table` 画了两行对照表。第一行是"推导式"构造的 `good=build_matrix(2,3)`：执行 `good[0][0]=7` 后，第 0 行是 `[7,0,0]`、第 1 行仍是 `[0,0,0]`，"行是否同一对象"列是 False。第二行是"重复引用"构造的 `bad=[[0]*3]*2`：同样执行 `bad[0][0]=7`，结果第 0 行 `[7,0,0]`、第 1 行**也是** `[7,0,0]`，"行是否同一对象"列是 True——同一对象，一荣俱荣。这张表就是本章核心论点的实验证据。

## 4. 核心代码：get_concatenation

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def get_concatenation(nums: list[int]) -> list[int]:
    return nums + nums
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

```python
def get_concatenation(nums: list[int]) -> list[int]:
    return nums + nums

def build_matrix(rows: int, cols: int) -> list[list[int]]:
    if rows < 0 or cols < 0:
        raise ValueError('dimensions must be nonnegative')
    return [[0 for _ in range(cols)] for _ in range(rows)]
```

**函数一：`get_concatenation`**

- `def get_concatenation(nums: list[int]) -> list[int]:` —— 接收一个整数列表，返回一个整数列表。类型注解 `list[int]` 读作"整数列表"。
- `return nums + nums` —— 整个函数就这一行。`+` 对列表的含义是"创建新列表并按序拼接"：它不修改 nums，而是返回一个长度翻倍的新对象。这满足"保留输入不变"的契约。（元素层面：新列表里每个位置抄的是原元素对象的引用；对整数这种不可变对象，抄引用完全安全。）

**函数二：`build_matrix`**

- `if rows < 0 or cols < 0:` / `raise ValueError('dimensions must be nonnegative')` —— 入口守卫：行数或列数为负是非法输入（`range` 遇到负数只会给空序列，静默返回空矩阵反而掩盖问题），直接报错说明原因。
- `return [[0 for _ in range(cols)] for _ in range(rows)]` —— 这是嵌套的两层列表推导式。读法从外往内：外层 `for _ in range(rows)` 循环 rows 次，每次迭代都要求值一次内层表达式 `[0 for _ in range(cols)]`——也就是**新建**一个含 cols 个 0 的列表作为外层的一个元素。正因为内层表达式每轮被重新执行，每一行都是刚出炉的新对象，互不共享。对比 `[[0]*cols]*rows`：那里内层只求值一次，之后只是把同一个引用抄了 rows 遍。两行代码长度差不多，语义天差地别，这就是本章要你刻在脑子里的区别。
- 下划线 `_` 是习惯写法，表示"这个循环变量我不会用到"，只关心循环次数。

**小实例表格的生成代码（“运行示例”部分）逐段讲解：**

（对应程序见下方分段实现与完整代码。）

- `from IPython.display import display, HTML` / `from html import escape` —— “运行示例”部分 先导入两个展示工具：display 和 HTML 负责"把拼好的字符串当网页元素渲染出来"，escape 负责"把任意值转成安全文本"（值里就算含有 `<` 这种字符，也不会被浏览器当成标签吃掉）。
- `def show_table(headers, rows):` —— 这是展示辅助函数：两个推导式分别把表头拼成 `<th>`、把每行数据拼成 `<td>`，再套上 `<table>` 骨架交给 display。它只管画表，不含任何算法；每个格子入库前都过一遍 `escape(str(x))`。
- `good=build_matrix(2,3)` —— 用我们自己的接口造一个 2 行 3 列的矩阵，`good[0]` 和 `good[1]` 是外层推导式两次迭代分别求值产生的两个不同列表。
- `bad=[[0]*3]*2` —— 对照组：经典错误写法。内层 `[0]*3` 只求值一次得到 `[0,0,0]`，外层 `*2` 把这一个引用抄了两遍，所以 `bad[0] is bad[1]` 为 True。
- `good[0][0]=7` / `bad[0][0]=7` —— 对两个矩阵做完全相同的操作：只把第 0 行第 0 个元素改成 7。这是对照实验里唯一的"自变量"，两种构造的其他条件全部一致。
- `show_table(['构造','第0行','第1行','行是否同一对象'], ...)` —— 把两种构造的四项观察排成两行：构造方式、第 0 行内容、第 1 行内容、两行身份检查的结果（`good[0] is good[1]` 和 `bad[0] is bad[1]`）。表格数据是内核现跑 `build_matrix` 算出来的，不是预制图片。

逐格读这张表（2×3 实验）：第一行"推导式"的第 0 行是 `[7,0,0]`、第 1 行是 `[0,0,0]`、身份列是 False——改第 0 行只动了第 0 行。第二行"重复引用"的第 0 行和第 1 行**都是** `[7,0,0]`、身份列是 True——`bad[1][0]` 变成 7 不是巧合的同步，而是 `bad[1]` 和 `bad[0]` 从头到尾就是同一个列表，改一次当然两边都看得见。这张两行表就是"别名事故"最直观的实验证据。

## 5. 分段实现：按顺序运行

**本章接口：** `get_concatenation(nums)`、`build_matrix(rows, cols)`

### get_concatenation

In [1]:
def get_concatenation(nums: list[int]) -> list[int]:
    return nums + nums

### build_matrix

In [2]:
def build_matrix(rows: int, cols: int) -> list[list[int]]:
    if rows < 0 or cols < 0:
        raise ValueError('dimensions must be nonnegative')
    return [[0 for _ in range(cols)] for _ in range(rows)]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

good=build_matrix(2,3)
bad=[[0]*3]*2
good[0][0]=7
bad[0][0]=7
show_table(['构造','第0行','第1行','行是否同一对象'], [('推导式',good[0],good[1],good[0] is good[1]),('重复引用',bad[0],bad[1],bad[0] is bad[1])])

构造,第0行,第1行,行是否同一对象
推导式,"[7, 0, 0]","[0, 0, 0]",False
重复引用,"[7, 0, 0]","[7, 0, 0]",True


## 7. 正确性、适用条件与复杂度

每个循环迭代新建一个内层列表，因此各行身份不同。拼接按原序各复制一次引用；对整数列表这满足无意外修改输入的契约。

**代价：** 拼接 O(n) 时间及输出空间；矩阵 O(rows×cols) 时间及输出空间。

### 展开理解

**拼接为什么对：** `nums + nums` 的语义由 Python 定义保证：新建列表，先按顺序放入左操作数的全部元素，再放入右操作数的全部元素。对本题来说就是 nums 的每个元素恰好出现两次、顺序与原数组一致，正好是"写两遍"的题意。同时这个操作从头到尾没对 nums 做任何原地修改，所以"输入保持不变"的契约自动满足；返回的是新对象，调用方改它也不会反过来影响原数组。

**矩阵为什么对：** 关键论点是"行身份"：外层推导式的**每次迭代都会新建一个内层列表**，所以 rows 个行是 rows 个互不相同的对象（`good[0] is good[1]` 是 False）。既然对象不同，对第 0 行的任何原地修改（比如 `good[0][0]=7`）就只落在第 0 行这一个对象身上，第 1 行毫无感知，仍保持全 0。每一行的内容又是 `[0 for _ in range(cols)]` 亲手生成的 cols 个 0，长度和初值都精确。两个维度都符合预期，所以整体正确。反过来，`[[0]*cols]*rows` 的所有行是同一个对象，改一处全体变，怎么测都是错的。

**复杂度是多少：** 拼接要把 2n 个元素抄进新列表，时间 O(n)、输出空间 O(n)（n=10⁵ 时大约复制 20 万个引用，瞬间完成；注意这 2n 的输出空间是题目要求的返回值本身，不算"额外"浪费）。矩阵要对 rows×cols 个格子各写一个 0，时间 O(rows×cols)，输出空间同样是 O(rows×cols)——比如 1000×1000 的矩阵就是一百万个格子，仍然毫无压力。

**共享不可变对象为什么没事：** `build_matrix` 产出的矩阵里，rows×cols 个格子装的都是指向整数 0 的引用（这些 0 完全可以共享同一个对象）。这看起来又是"共享引用"，为什么不翻车？因为整数不可变：你没办法"原地"把 0 改成 7，赋值 `m[0][0]=7` 做的事是把**行列表**（可变容器）里位置 0 的那个引用换成指向 7，其他格子的引用原封不动。对照 `bad` 的事故：那里共享的是可变的行列表本身，改的是共享对象的内容，所以全体遭殃。判断标准一句话：**共享不可变对象安全，共享可变对象危险**。

**换一种实现方式，量级会怎么变：** 练习 2 让你不用 `+` 生成双倍数组。如果用 `ans=[]` 加 for 循环 append 两遍，总代价仍是 O(n)（2n 次 append 摊还下来每次常数）；但如果有人写成 `ans = ans + [x]` 逐个拼接，第 k 步都要把已有结果整体复制一遍，对 2n 个元素总计约 1+2+…+2n ≈ 2n² 次引用复制——n=10⁵ 时约 200 亿次；而 `nums + nums` 一次批量复制 2n=20 万个引用，差了五个数量级。同一道题，写法一变就从"瞬间完成"变成"大概率超时"。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的代码逐条看：

```python
a = [1, 2, 1]

b = get_concatenation(a)

assert b == [1, 2, 1, 1, 2, 1] and a == [1, 2, 1] and (b is not a)

m = build_matrix(2, 2)

m[0][0] = 1

assert m[1][0] == 0 and m[0] is not m[1]

assert build_matrix(0, 3) == []

print('N005: 所有本章断言通过')
```

- `a=[1,2,1]` / `b=get_concatenation(a)`：准备输入并调用。选 `[1,2,1]` 正是 LeetCode 1929 的示例输入，期望输出 `[1,2,1,1,2,1]`。
- `assert b == [1,2,1,1,2,1] and a == [1,2,1] and b is not a`：一条断言同时查三件事。`b == [1,2,1,1,2,1]` 查返回值内容正确（正常值测试）；`a == [1,2,1]` 查**输入没被改坏**——如果你用原地 append 实现（比如 `a.extend(a)` 再返回 a），a 会变成 `[1,2,1,1,2,1]`，这条立刻失败；`b is not a` 查返回的是**新对象**而非原数组本身（身份测试，`==` 看内容、`is` 看身份，两者要分清）。
- `m=build_matrix(2,2)` / `m[0][0]=1`：造一个 2×2 矩阵并做"危险操作"——只改第 0 行第 0 个元素。
- `assert m[1][0] == 0 and m[0] is not m[1]`：两个检查合在一起正是"行独立"的验收标准：改完 `m[0][0]` 后 `m[1][0]` 必须还是 0（内容不受牵连），并且 `m[0] is not m[1]`（两行根本不是同一个对象）。如果你用 `*rows` 写法实现，这两个条件全崩，这条断言直接暴露 bug。
- `assert build_matrix(0,3) == []`：零行边界。rows=0 时外层推导式循环零次，返回空列表；验证矩阵在退化维度下的行为，不报错、不返回奇怪的东西。
- 最后的 print：前面全部通过才会打印，代表整组测试通过。

**如果实现写错了，哪条断言会先响？** 做个故障推演，帮你建立"断言 = 报警器"的直觉：

- 把 `get_concatenation` 写成原地版（`nums.extend(nums)` 再 `return nums`）：第一条断言立刻挂——`a == [1,2,1]` 不成立（a 已被改成六个元素），`b is not a` 也不成立（返回的就是 a 本人）。返回内容碰巧是对的，但"不改输入、返回新对象"两条契约全碎，这条断言正是为这种实现准备的。
- 把 `build_matrix` 写成 `[[0]*cols]*rows`：`m[0][0]=1` 之后 `m[1][0]` 也是 1，`m[1][0] == 0` 与 `m[0] is not m[1]` 两个条件同时崩，第二条断言当场暴露 bug。
- 把入口守卫删掉再调用 `build_matrix(-1,3)`：`range(-1)` 是空序列，外层循环零次，函数**静默**返回 `[]` 而不报错——这正是守卫存在的意义：把"负维度"这种坏输入从"悄悄给个怪结果"变成"大声报错"。“自动测试”部分 没有覆盖负数输入，这条属于你做 “掌握检查”部分 自测时应该自己补上的用例。

**把三条断言分个层：**

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `b == [1,2,1,1,2,1] and a == [1,2,1] and b is not a` | 正常值 + 契约 | 内容对、输入不动、返回新对象 |
| `m[1][0] == 0 and m[0] is not m[1]` | 危险操作后验 | 改一行不牵连另一行 |
| `build_matrix(0,3) == []` | 边界 | 零行维度安静退化 |

In [4]:
a = [1, 2, 1]

b = get_concatenation(a)

assert b == [1, 2, 1, 1, 2, 1] and a == [1, 2, 1] and (b is not a)

m = build_matrix(2, 2)

m[0][0] = 1

assert m[1][0] == 0 and m[0] is not m[1]

assert build_matrix(0, 3) == []

print('N005: 所有本章断言通过')

N005: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 修复[[0]*m]*n。

**练习 2：** 不使用加号生成双倍数组。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（修复 `[[0]*m]*n`）**：提示——先复现事故：构造 `bad=[[0]*3]*2`，执行 `bad[0][0]=7`，打印整个 bad 观察两行同时变 7；再用 `bad[0] is bad[1]` 证实两行是同一对象。修复方向就是把"乘法复制引用"换成"循环重新造行"：可以用嵌套推导式（本章 `build_matrix` 的写法），也可以用 for 循环每轮 `rows.append([0]*cols)`。要点是解释清楚"`*n` 复制的是引用不是对象"。手算示例用 2×3，边界用 0 行或 0 列（期望 `[]` 或 `[[]]`）。
- **练习 2（不使用加号生成双倍数组）**：提示——目标是得到与 `nums + nums` 相同的结果但不用 `+`。方向一：建一个空列表 `ans`，用 for 循环跑两遍（`for _ in range(2): for x in nums: ans.append(x)`），注意保持原序；方向二：直接 `ans = nums.copy()` 然后再逐个 append，或者用推导式 `[x for _ in range(2) for x in nums]`。验收时除了内容相等，别忘了补上 `a == [1,2,1]` 和 `ans is not a` 两条契约检查，保持"不改输入、返回新对象"。手算示例仍用 `[1,2,1]`，期望 `[1,2,1,1,2,1]`。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def get_concatenation(nums: list[int]) -> list[int]:
    return nums + nums

def build_matrix(rows: int, cols: int) -> list[list[int]]:
    if rows < 0 or cols < 0:
        raise ValueError('dimensions must be nonnegative')
    return [[0 for _ in range(cols)] for _ in range(rows)]

# 示例与回归测试
a = [1, 2, 1]

b = get_concatenation(a)

assert b == [1, 2, 1, 1, 2, 1] and a == [1, 2, 1] and (b is not a)

m = build_matrix(2, 2)

m[0][0] = 1

assert m[1][0] == 0 and m[0] is not m[1]

assert build_matrix(0, 3) == []

print('N005: 所有本章断言通过')

N005: 所有本章断言通过


## 11. 代表题与迁移

- **1929. Concatenation of Array**：数组拼接原题，练的是"`+` 创建新列表 vs 原地修改"的区分，以及"返回值是新对象、输入不动"的契约意识，对应本章 `get_concatenation`。
- **1920. Build Array from Permutation**：给排列 nums，返回 `ans[i] = nums[nums[i]]`。它练的是本章的**嵌套索引**（`nums[nums[i]]` 这种一层套一层的下标访问）和"读输入构造新输出、不改输入"的模式，正好是 `get_concatenation` 契约的进阶版。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。